# Kiểm chứng dữ liệu, luật và model đã lưu
Notebook chỉ đọc kết quả, tính lại để đối chiếu và hiển thị kiểm tra; không train, không chọn lại cấu hình, không sinh thêm báo cáo.

In [1]:
from pathlib import Path
from itertools import combinations, product
from datetime import datetime, timezone
import argparse
import hashlib
import json
import math

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'project_manifest.json').exists())
OUT = ROOT / 'results/arm'
DATA_OUT = ROOT / 'data/arm'
HORIZONS = (1, 5, 10, 20)
FEATURES = [
    'Gold_Daily_Return', 'Gold_Return_Lag_1', 'Gold_Intraday_Range',
    'Gold_Volatility_20', 'Gold_Distance_to_MA_20', 'Gold_RSI_14',
    'Gold_MACD_Histogram', 'Oil_Price_Change', 'Dollar_Daily_Return',
    'SP500_Daily_Return', 'SP500_Drawdown',
]
QUANTILE_FEATURES = ['Gold_Intraday_Range', 'Gold_Volatility_20',
                     'Gold_RSI_14', 'SP500_Drawdown']
GRID = list(product((0.02, 0.05, 0.10), (0.50, 0.55, 0.60), (1.00, 1.05, 1.10)))
TARGET = 'Gold_Target_Next'
MIN_VAL_COUNT = 30


import pickle
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, log_loss, brier_score_loss, accuracy_score, balanced_accuracy_score
from IPython.display import display
torch.set_num_threads(2)

## 1. Kiểm chứng ARM

In [2]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_data(h):
    source = ROOT / f'data/modeling/horizons/h{h}'
    data = pd.read_csv(source / 'modeling_dataset.csv', parse_dates=['Date'])
    trace = pd.read_csv(source / 'target_trace.csv', parse_dates=['Date', 'Target_Date'])
    data = data.merge(trace[['Date', 'Target_Date', 'Close', 'Target_Close']],
                      on='Date', validate='one_to_one')
    if data[FEATURES + [TARGET]].isna().any().any() or data.Date.duplicated().any():
        raise ValueError('Missing feature/target or duplicate date')
    if not data.Date.is_monotonic_increasing or not data.Target_Date.gt(data.Date).all():
        raise ValueError('Invalid date order')
    if not data[TARGET].eq((data.Target_Close > data.Close).astype(int)).all():
        raise ValueError('Target disagrees with price trace')
    return data

def split_masks(data):
    # Purge labeled endpoints only; ARM does not consume recurrent windows.
    return {
        'train': (data.Date < '2021-01-01') & (data.Target_Date < '2021-01-01'),
        'val': data.Date.between('2021-01-01', '2023-12-31') & (data.Target_Date < '2024-01-01'),
        'test': data.Date >= '2024-01-01',
    }
def fit_thresholds(train):
    rows = []
    for name in QUANTILE_FEATURES:
        q1, q2 = train[name].quantile([1/3, 2/3], interpolation='linear')
        rows.append({'Feature': name, 'q1': float(q1), 'q2': float(q2),
                     'Train_Rows': len(train), 'Interpolation': 'linear'})
    return pd.DataFrame(rows)

def encode(frame, thresholds, h):
    """Encode original unscaled values; quantile edges are fitted on Train only."""
    encoded = frame[['Date', 'Target_Date']].copy()
    cuts = thresholds.set_index('Feature')
    for name in FEATURES:
        values = frame[name]
        if name in QUANTILE_FEATURES:
            states = ['Deep', 'Medium', 'Shallow'] if name == 'SP500_Drawdown' else ['Low', 'Medium', 'High']
            q1, q2 = cuts.loc[name, ['q1', 'q2']]
            state = np.select([values <= q1, values <= q2], states[:2], default=states[2])
        else:
            # Zero is exact equality, without a near-zero tolerance.
            state = np.select([values < 0, values == 0], ['Negative', 'Zero'], default='Positive')
        encoded[name] = name + '=' + pd.Series(state, index=frame.index)
    encoded['Target_Item'] = np.where(frame[TARGET] == 1, f'Gold_Up_H{h}', f'Gold_NonUp_H{h}')
    return encoded

def item_names(h):
    names = []
    for name in FEATURES:
        states = (['Deep', 'Medium', 'Shallow'] if name == 'SP500_Drawdown'
                  else ['Low', 'Medium', 'High'] if name in QUANTILE_FEATURES
                  else ['Negative', 'Zero', 'Positive'])
        names.extend(f'{name}={state}' for state in states)
    return names + [f'Gold_NonUp_H{h}', f'Gold_Up_H{h}']

def onehot(encoded, h):
    matrix = {}
    for item in item_names(h):
        field = item.split('=')[0] if '=' in item else 'Target_Item'
        matrix[item] = encoded[field].eq(item).to_numpy()
    return pd.DataFrame(matrix, index=encoded.index)
def measure_rules(rules, matrix, prefix):
    """Support is P(antecedent AND consequent), not antecedent coverage."""
    rows = []
    for rule in rules.itertuples(index=False):
        antecedent = matrix[rule.Antecedent.split(' & ')].all(axis=1).to_numpy()
        consequent = matrix[rule.Consequent].to_numpy()
        n_a = int(antecedent.sum())
        n_joint = int((antecedent & consequent).sum())
        base = float(consequent.mean())
        conf = n_joint / n_a if n_a else np.nan
        rows.append({f'{prefix}_Rows': len(matrix), f'{prefix}_Antecedent_Count': n_a,
                     f'{prefix}_Joint_Count': n_joint, f'{prefix}_Coverage': n_a / len(matrix),
                     f'{prefix}_Support': n_joint / len(matrix), f'{prefix}_Base_Rate': base,
                     f'{prefix}_Confidence': conf, f'{prefix}_Lift': conf / base if base else np.nan,
                     f'{prefix}_Leverage': n_joint / len(matrix) - n_a / len(matrix) * base})
    return pd.concat([rules.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

def check(condition,message):
    if not condition:
        raise ValueError(message)


In [3]:
checks = []
for h in HORIZONS:
    state = pickle.loads((DATA_OUT/f'ARM-H{h}/train_encoding.pkl').read_bytes())
    data = load_data(h)
    masks = split_masks(data)
    cuts = fit_thresholds(data.loc[masks['train']])
    pd.testing.assert_frame_equal(cuts,state['thresholds'])
    candidates = pd.read_csv(OUT/f'1_train/H{h}_rules.csv')
    selected = pd.read_csv(OUT/f'2_validation/H{h}_selected_rules.csv')
    tested = pd.read_csv(OUT/f'3_test/H{h}_evaluated_rules.csv')
    check(set(selected.Rule_ID)==set(tested.Rule_ID),'Test must evaluate every selected rule')
    chosen = selected.iloc[0]
    expected = candidates.loc[
        candidates.Train_Support.ge(chosen.min_support) &
        candidates.Train_Confidence.ge(chosen.min_confidence) &
        candidates.Train_Lift.ge(chosen.min_lift) & candidates.Val_Stable]
    check(set(expected.Rule_ID)==set(selected.Rule_ID),'Validation selection differs')
    for split,prefix,table in [('train','Train',candidates),('val','Val',candidates),('test','Test',tested)]:
        matrix = onehot(encode(data.loc[masks[split]],cuts,h),h)
        clean = table.drop(columns=[c for c in table if c.startswith(prefix+'_')])
        measured = measure_rules(clean,matrix,prefix)
        for metric in ['Support','Confidence','Lift','Antecedent_Count','Joint_Count']:
            column = f'{prefix}_{metric}'
            check(np.allclose(table[column],measured[column],equal_nan=True),f'H{h} {column} mismatch')
    checks.append({'Horizon':h,'Train':len(candidates),'Validation':len(selected),'Test':len(tested),'Status':'PASS'})
print('Kiểm tra ARM Train/Val/Test:'); display(pd.DataFrame(checks))


Kiểm tra ARM Train/Val/Test:


,Horizon,Train,Validation,Test,Status
0,1,4374,835,835,PASS
1,5,4341,476,476,PASS
2,10,4328,442,442,PASS
3,20,4335,536,536,PASS


In [4]:
print('ARM: PASS. Không lưu JSON kiểm chứng trong results/arm.')

ARM: PASS. Không lưu JSON kiểm chứng trong results/arm.


## 2. Nạp lại model classification và kiểm tra Test
Kiểm tra cả scaler fit đúng Train, ngưỡng 0.5 và dự đoán khớp file đã lưu, không train lại.

In [5]:
class SequenceClassifier(nn.Module):
    def __init__(self,inputs,algorithm):
        super().__init__()
        self.recurrent=(nn.RNN(inputs,32,num_layers=1,nonlinearity='tanh',batch_first=True)
                        if algorithm=='RNN' else nn.LSTM(inputs,32,num_layers=1,batch_first=True))
        self.dropout=nn.Dropout(0.2);self.head=nn.Linear(32,1)
    def forward(self,x):
        sequence,_=self.recurrent(x)
        return self.head(self.dropout(sequence[:,-1,:]))

In [6]:
def probabilities(model, x):
    model.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(model(batch)) for (batch,) in DataLoader(TensorDataset(x),batch_size=256)]).numpy().ravel()

def calibration(truth, probability):
    # Ten equal-width bins [0,.1), ... [.9,1], without dropping empty bins.
    ids = np.minimum((np.asarray(probability)*10).astype(int),9)
    rows=[]
    for index in range(10):
        mask=ids==index
        count=int(mask.sum())
        rows.append({'Bin':index,'Lower':index/10,'Upper':(index+1)/10,'Count':count,
                     'Mean_Probability_Up':float(np.asarray(probability)[mask].mean()) if count else None,
                     'Observed_Up_Rate':float(np.asarray(truth)[mask].mean()) if count else None})
    table=pd.DataFrame(rows)
    ece=float(((table.Count/len(truth))*(table.Mean_Probability_Up-table.Observed_Up_Rate).abs().fillna(0)).sum())
    return table,ece

def scores(truth, probability):
    pred=np.asarray(probability)>=0.5
    _,ece=calibration(truth,probability)
    return {'ROC_AUC':roc_auc_score(truth,probability),
            'Macro_F1':f1_score(truth,pred,average='macro',zero_division=0),
            'Log_Loss':log_loss(truth,probability,labels=[0,1]),
            'Brier_Score':brier_score_loss(truth,probability), 'ECE_10':ece,
            'Accuracy':accuracy_score(truth,pred),'Balanced_Accuracy':balanced_accuracy_score(truth,pred),
            'Pred_Up':int(pred.sum()),'Pred_Non_Up':int((~pred).sum())}

In [7]:
classification_checks = []
for h in HORIZONS:
    data = load_data(h)
    all_features = FEATURES
    for variant in ['A','B']:
        names = [name for name in all_features if name.startswith('Gold_')] if variant=='A' else all_features
        for algorithm in ['Naive','RF','LR','RNN','LSTM']:
            out = ROOT/f'results/classification/h{h}/model_{variant}'
            is_sequence = algorithm in ['RNN','LSTM']
            boundary = '2021-01-01' if is_sequence else '2024-01-01'
            train_mask = (data.Date < boundary) & (data.Target_Date < boundary)
            label = {'RF':'RandomForest','LR':'LogisticRegression'}.get(algorithm,algorithm)
            bundle = pickle.loads((out/f'{label}_model_and_scaler.pkl').read_bytes())
            scaler = bundle['scaler']
            raw_train = data.loc[train_mask,names].to_numpy()
            check(np.allclose(scaler.mean_,raw_train.mean(axis=0)), 'Scaler mean uses incorrect Train')
            check(np.allclose(scaler.var_,raw_train.var(axis=0)), 'Scaler variance uses incorrect Train')
            transformed = scaler.transform(data[names])
            test_positions = np.flatnonzero(data.Date.ge('2024-01-01').to_numpy())
            frame = data.iloc[test_positions]
            saved = pd.read_csv(out/f'{label}_test_predictions.csv',parse_dates=['Date','Target_Date'])
            check(saved.Date.tolist()==frame.Date.tolist(), 'Test dates differ')
            check(saved.Actual.tolist()==frame[TARGET].tolist(), 'Test target differs')
            if is_sequence:
                windows = np.stack([transformed[i-19:i+1] for i in test_positions]).astype('float32')
                model = SequenceClassifier(len(names),algorithm)
                model.load_state_dict(bundle['state_dict'])
                probability = probabilities(model,torch.from_numpy(windows))
            else:
                model = bundle['model']
                probability = model.predict_proba(transformed[test_positions])[:,list(model.classes_).index(1)]
            check(np.allclose(probability,saved.Probability_Up,atol=1e-7), 'Reloaded prediction mismatch')
            check(np.array_equal(probability>=.5,saved.Predicted), 'Threshold differs from 0.5')
            actual_metrics = scores(saved.Actual,probability)
            comparison = pd.read_csv(ROOT/'results/classification/test_model_comparison.csv')
            recorded = comparison.loc[comparison.Horizon.eq(h)&comparison.Variant.eq(variant)&comparison.Model.eq(algorithm)].iloc[0]
            for name,value in actual_metrics.items():
                check(np.isclose(value,recorded[name],atol=1e-7), f'Metric mismatch: {name}')
            classification_checks.append({'Horizon':h,'Variant':variant,'Model':algorithm,'Test_Rows':len(frame),'Status':'PASS'})
print('Kiểm chứng 40 model/scaler và dự đoán Test:')
display(pd.DataFrame(classification_checks))


Kiểm chứng 40 model/scaler và dự đoán Test:


,Horizon,Variant,Model,Test_Rows,Status
0,1,A,Naive,683,PASS
1,1,A,RF,683,PASS
2,1,A,LR,683,PASS
3,1,A,RNN,683,PASS
4,1,A,LSTM,683,PASS
5,1,B,Naive,683,PASS
6,1,B,RF,683,PASS
7,1,B,LR,683,PASS
8,1,B,RNN,683,PASS
9,1,B,LSTM,683,PASS


## 3. Code local và cấu trúc
Không chấp nhận phụ thuộc xử lý vào src cũ, Colab hay các project nguồn. Mọi thư mục đều cần README.

In [8]:
import nbformat
notebook_checks = []
for path in sorted((ROOT/'notebook').rglob('*.ipynb')):
    notebook = nbformat.read(path,as_version=4)
    for index,cell in enumerate(notebook.cells):
        if cell.cell_type != 'code':
            continue
        compile(cell.source,f'{path.name}:cell{index}','exec')
        for forbidden in ['from horizon_experiments import','import arm_horizons','from rnn_classification import','from lstm_shallow_classification import','from google.colab','/content/drive']:
            check(path.name == '01_verify_project.ipynb' or forbidden not in cell.source,f'External dependency remains: {path.name}')
    notebook_checks.append({'Notebook':str(path.relative_to(ROOT)),'Code':'PASS'})
missing_readmes = [str(p.relative_to(ROOT)) for p in ROOT.rglob('*') if p.is_dir() and not (p/'README.md').exists()]
check(not missing_readmes,f'Folders without README: {missing_readmes}')
print('Notebook code và README:')
display(pd.DataFrame(notebook_checks))


D:\1_TaiLieuHocTap\2627\DataMining\Project\.venv_eda\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


Notebook code và README:


,Notebook,Code
0,notebook\00_prepare_data.ipynb,PASS
1,notebook\01_verify_project.ipynb,PASS
2,notebook\ARM\ARM-FINAL\arm_rule_discovery.ipynb,PASS
3,notebook\ARM\ARM-FINAL\arm_test.ipynb,PASS
4,notebook\ARM\ARM-H1\arm_h1.ipynb,PASS
5,notebook\ARM\ARM-H10\arm_h10.ipynb,PASS
6,notebook\ARM\ARM-H20\arm_h20.ipynb,PASS
7,notebook\ARM\ARM-H5\arm_h5.ipynb,PASS
8,notebook\Classification\H1\classification_A.ipynb,PASS
9,notebook\Classification\H1\classification_B.ipynb,PASS
